# Vectors, Matrices & Operations

### Learning objectives 

 1. Build a Matrix class with element-wise operaitons, matrix multiplication, transpose, determinant, and inverse
 2. Distinguise element-wise multiplication from matrix multiplicaiton and explain when each applies
 3. Implement a single dense neural network layer(```relu(W @ X + b) ```) using only the from scratch Matrix class
 4. Explain broadcasting rules and how bias addition works in neural network frameworks 

In [3]:
class Vector:
    def __init__(self,data):
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self):
        return f"Vector({self.data})"

    def __add__(self,other):
        return Vector([ a+b for a,b in zip(self.data,other.data)])

    def __sub__(self,other):
        return Vector([ a - b for a,b in zip(self.data,other.data)])
    def __mul__(self,scalar):
        return Vector([x * scalar for x in self.data])
    def dot(self,other):
        return sum(a * b for a,b in zip(self.data, other.data))

    def magnitude(self):
        return sum(x ** 2 for x in self.data) ** 0.5
    

In [19]:
class Matrix:
    def __init__(self,data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows,self.cols)

    def __repr__(self):
        rows_str = "\n ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n {rows_str}"
    def __add__(self,other):
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def __sub__(self,other):
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)])
    def element_wise_multiply(self, other):
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def matmul(self,other):
        return Matrix([
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])
    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)])
    def determinant(self):
        if self.shape == (1,1):
            return self.data[0][0]
        if self.shape == (2,2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k!=j]
                for i in range(1,self.rows)
            ])
            det  += (-1 ** j) * self.data[0][j] * minor.determinant()
        return det
    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix([
        [self.data[1][1] / det, -self.data[0][1] / det],
        [-self.data[1][0] / det, self.data[0][0] / det]
        ])
    @staticmethod
    def identity(n):
        return Matrix([
            [1 if i == j else 0 for j in range(n)]
            for i in range(n)])
        

In [20]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print("A + B =", (A + B).data)
print("A @ B =", A.matmul(B).data)
print("A^T =", A.transpose().data)
print("det(A) =", A.determinant())
print("A^-1 =", A.inverse_2x2().data)

I = Matrix.identity(2)
print("A @ A^-1 =", A.matmul(A.inverse_2x2()).data)

A + B = [[6, 8], [10, 12]]
A @ B = [[19, 22], [43, 50]]
A^T = [[1, 3], [2, 4]]
det(A) = -2
A^-1 = [[-2.0, 1.0], [1.5, -0.5]]
A @ A^-1 = [[1.0, 0.0], [0.0, 1.0]]


In [21]:
import random 

inputs = Matrix([[0.5] , [0.8] , [0.2]])
weights = Matrix([
    [random.uniform(-1,1) for _ in range(3)]
    for _ in range(2)])
bias = Matrix([[0.1], [0.1]])

def relu_matrix(m):
    return Matrix([[max(0,val) for val in row] for row in m.data])
pre_activation = weights.matmul(inputs) + bias
output = relu_matrix(pre_activation)

print(f"Input shape: {inputs.shape}")
print(f"Weight shape: {weights.shape}")
print(f"Output shape: {output.shape}")
print(f"Output: {output.data}")

Input shape: (3, 1)
Weight shape: (2, 3)
Output shape: (2, 1)
Output: [[0.49030218006337956], [0.0031472768199558576]]


## Now with numpy 

In [22]:
import numpy as np 
A = np.array([[1,2],[3,4]])
B = np.array([[5,6],[7,8]])

print("A + B =\n", A + B)
print("A * B (element-wise) =\n", A * B)
print("A @ B (matrix multiply) =\n", A @ B)
print("A^T =\n", A.T)
print("det(A) =", np.linalg.det(A))
print("A^-1 =\n", np.linalg.inv(A))
print("I = \n", np.eye(2))

inputs = np.random.randn(3,1)
weights = np.random.randn(2,3)
bias = np.array([[0.1], [0.1]])
output = np.maximum(0, weights @ inputs + bias)

print(f"\nNeural network layer: {weights.shape} @ {inputs.shape} = {output.shape}")
print(f"Ouput: \n{output}")


A + B =
 [[ 6  8]
 [10 12]]
A * B (element-wise) =
 [[ 5 12]
 [21 32]]
A @ B (matrix multiply) =
 [[19 22]
 [43 50]]
A^T =
 [[1 3]
 [2 4]]
det(A) = -2.0000000000000004
A^-1 =
 [[-2.   1. ]
 [ 1.5 -0.5]]
I = 
 [[1. 0.]
 [0. 1.]]

Neural network layer: (2, 3) @ (3, 1) = (2, 1)
Ouput: 
[[0.]
 [0.]]


## Broadcasting with Numpy 
 **Note**: Numpy automatically broadcasts the 1D bias across both rows. This is how bias addition works in every neural network framework 

In [23]:
matrix = np.array([[1,2,3] , [4,5,6]])
bias = np.array([10,20,30])
print(matrix + bias)

[[11 22 33]
 [14 25 36]]


### Exercises 

 **Exercise#1**:Verify the inverse. Multiply A @ A.inverse_2x2() and confirm you get the identity matrix. Try it with three different 2x2 matrices. What happens when the determinant is Zero?

In [29]:
# Matrix 1
A1 = Matrix([[4, 7], [2, 6]])
identity1 = A1.matmul(A1.inverse_2x2()).data
print("Matrix 1: A @ A^-1 =", identity1)

# Matrix 2 (create your own)
A2 = Matrix([[5,4], [9,13]])
identity2 = A2.matmul(A2.inverse_2x2()).data
print("Matrix 2: A @ A^-1 =", identity2)

# Matrix 3 (create your own)
A3 = Matrix([[12,13], [14,15]])
identity3 = A3.matmul(A3.inverse_2x2()).data
print("Matrix 3: A @ A^-1 =", identity3)

# Then try a singular matrix (det = 0)
A_singular = Matrix([[1, 2], [2, 4]])  # Linearly dependent rows
try:
    A_singular.inverse_2x2()
except ValueError as e:
    print(f"Singular matrix error: {e}")


Matrix 1: A @ A^-1 = [[0.9999999999999998, 4.440892098500626e-16], [-2.220446049250313e-16, 1.0000000000000004]]
Matrix 2: A @ A^-1 = [[0.9999999999999998, 0.0], [0.0, 1.0000000000000002]]
Matrix 3: A @ A^-1 = [[1.0, 0.0], [0.0, 1.0]]
Singular matrix error: Matrix is singular, no inverse exists


 **Exercise#2**:Extend the Matrix class to compute inverse for 3x3 matrices using the addjugate method. Test it against NumPy's ```np.linalg.inv```.

 1. First i will implement the co-factor method for position (i,j)
 2. Now implement it for the whole matrix
 3. Transpose the matrix to get the adjugagte matrix
 4. Lastly Scalar multiply the matrix to get the inverse

In [51]:
class Matrix:
    def __init__(self,data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows,self.cols)

    def __repr__(self):
        rows_str = "\n ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n {rows_str}"
    def __add__(self,other):
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def __sub__(self,other):
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)])
    def element_wise_multiply(self, other):
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def matmul(self,other):
        return Matrix([
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])
    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)])
    def determinant(self):
        if self.shape == (1,1):
            return self.data[0][0]
        if self.shape == (2,2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k!=j]
                for i in range(1,self.rows)
            ])
            det  += ((-1) ** j) * self.data[0][j] * minor.determinant()
        return det
    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix([
        [self.data[1][1] / det, -self.data[0][1] / det],
        [-self.data[1][0] / det, self.data[0][0] / det]
        ])
    @staticmethod
    def identity(n):
        return Matrix([
            [1 if i == j else 0 for j in range(n)]
            for i in range(n)])
    def cofactor(self,row,col):
        """Compute the cofactor at position (row, col) """
        minor_data = [
        [self.data[i][j] for j in range(self.cols) if j!=col]
        for i in range(self.rows) if i!= row ]
        minor = Matrix(minor_data)
        sign = (-1) ** (row + col)
        return sign * minor.determinant()
    def cofactor_matrix(self):
        """Compute the full cofactor matrix"""
        return Matrix([
            [self.cofactor(i,j) for j in range(self.cols)]
            for i in range(self.rows)])
    def Adjugate(self):
        """ Compute the Adjugate by transposing the matrix """
        cof = self.cofactor_matrix()
        adj = cof.transpose()
        return adj

    def inverse_3x3(self):
        """Compute 3x3 matrix inverse using adjugate method"""
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        adj = self.Adjugate()
        return adj.scalar_multiply(1 / det)

In [52]:
A = Matrix([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
print("Cofactor at (0,0):", A.cofactor(0, 0))  # Should be -24
print("Cofactor at (0,1):", A.cofactor(0, 1))  # Should be 20

Cofactor at (0,0): -24
Cofactor at (0,1): 20


In [53]:
A = Matrix([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
cof = A.cofactor_matrix()
print("Cofactor matrix:")
print(cof)

Cofactor matrix:
Matrix((3, 3)):
 [-24, 20, -5]
 [18, -15, 4]
 [5, -4, 1]


In [54]:
A = Matrix([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
adj = A.Adjugate()
print("Adjugate matrix")
print(adj)

Adjugate matrix
Matrix((3, 3)):
 [-24, 18, 5]
 [20, -15, -4]
 [-5, 4, 1]


In [55]:
A = Matrix([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
A_inv = A.inverse_3x3()
print("A^(-1):")
print(A_inv)

# Verify: A @ A^(-1) should be identity
identity = A.matmul(A_inv)
print("\nA @ A^(-1):")
print(identity)

A^(-1):
Matrix((3, 3)):
 [-24.0, 18.0, 5.0]
 [20.0, -15.0, -4.0]
 [-5.0, 4.0, 1.0]

A @ A^(-1):
Matrix((3, 3)):
 [1.0, 0.0, 0.0]
 [0.0, 1.0, 0.0]
 [0.0, 0.0, 1.0]


In [56]:
import numpy as np

A = Matrix([[1, 2, 3], [0, 1, 4], [5, 6, 0]])
A_inv = A.inverse_3x3()

# NumPy inverse
A_np = np.array([[1, 2, 3], [0, 1, 4], [5, 6, 0]], dtype=float)
A_inv_np = np.linalg.inv(A_np)

print("Our A_inv:")
print(A_inv.data)
print("\nNumPy A_inv:")
print(A_inv_np)

# Check if they match
print("\nAre they close?", np.allclose(A_inv.data, A_inv_np))

Our A_inv:
[[-24.0, 18.0, 5.0], [20.0, -15.0, -4.0], [-5.0, 4.0, 1.0]]

NumPy A_inv:
[[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]

Are they close? True


**Exercise#3**: Build a two-layer network. Using only your matri